In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA

load_dotenv(override=True)

key = os.getenv("OPENROUTER_API_KEY")
if key:
    print("API key loaded!")
else:
    print("Key not found - check .env file!")

print("All imports done!")

In [ ]:
loader = PyPDFLoader("Tech_and_AI_Report_2025.pdf")
pages = loader.load()
print(f"PDF loaded! Pages: {len(pages)}")

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
splits = splitter.split_documents(pages)
print(f"Chunks created: {len(splits)}")

In [ ]:
print("Loading embeddings... please wait...")
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)
print("Embeddings ready!")

In [ ]:
vectorstore = FAISS.from_documents(
    documents=splits,
    embedding=embeddings
)
print("Vector database ready!")

In [ ]:
llm = ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0
)
print("LLM ready!")

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever
)
print("Q/A Chain ready! Ask your questions below!")

In [ ]:
query = "Define Cybersecurity in the AI Era"
response = qa_chain.invoke(query)
print("Question:", query)
print("\nAnswer:", response["result"])